# Formula-SWCT Generalization on DermaMNIST

This notebook evaluates the final Formula-SWCT method on DermaMNIST from the MedMNIST benchmark using an ImageNet-pretrained ResNet-18.

DermaMNIST introduces a medical dermatology classification setting that differs substantially from the natural-image datasets used during Formula-SWCT calibration.

The evaluation compares:

- Baseline frozen ResNet-18,
- original Single-Parameter Curvature Tuning (S-CT),
- Formula-SWCT.

Formula-SWCT follows the predict-first, train-once protocol:

1. Fit the Formula-SWCT predictor using only the calibration datasets.
2. Collect stage statistics from the DermaMNIST training split.
3. Predict four stage-specific beta values before downstream training.
4. Freeze the predicted beta values.
5. Extract fixed train, validation, and test features once.
6. Train one downstream linear classifier.
7. Select the classifier checkpoint using validation accuracy.
8. Evaluate once on the test split.

No DermaMNIST-specific beta sweep or beta optimization is performed by Formula-SWCT.

## Environment Setup

This notebook is designed to run independently in a fresh Google Colab runtime.

The repository is cloned, the `stage-wise-ct` branch is selected, and the required research dependencies are installed.

After dependency installation, restart the runtime before continuing.

In [ ]:
!nvidia-smi

%cd /content

import os

if not os.path.exists("/content/curvature-tuning-research"):
    !git clone -b stage-wise-ct https://github.com/Syed1611/curvature-tuning-research.git
else:
    print("Repository already exists; skipping clone.")

%cd /content/curvature-tuning-research
!git checkout stage-wise-ct
!git pull

%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

In [ ]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

### Runtime Restart Required

Restart the Colab runtime after dependency installation.

After restarting, do not rerun the installation cell. Continue from the next cell.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())

In [ ]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

## DermaMNIST Dataset Setup

DermaMNIST is loaded through the MedMNIST dataset integration in the project's transfer-learning pipeline.

DermaMNIST contains seven skin-lesion classes and provides a medical-domain transfer setting substantially different from the natural-image datasets used for Formula-SWCT calibration.

This cell verifies the train, validation, and test splits before running the evaluation methods.

In [ ]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_medmnist/dermamnist",
    train_batch_size=32,
    test_batch_size=64,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Label min:", labels.min().item())
print("Label max:", labels.max().item())

# Baseline and Original S-CT Reference

The Baseline and original Single-Parameter Curvature Tuning methods are evaluated on DermaMNIST using seeds 42, 43, and 44.

The Baseline uses the frozen ImageNet-pretrained ResNet-18 with its original ReLU activations.

S-CT performs its original validation-based search for one global beta value shared across the network.

These three runs provide the reference against which Formula-SWCT is evaluated.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

/content/curvature-tuning-research/src/curvature-tuning
2026-09-28 20:45:36.257 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet18_seed42.log
2026-09-28 20:45:36.260 | INFO     | __main__:main:58 - Running on cuda
2026-09-28 20:45:42.836 | INFO     | __main__:main:85 - Testing baseline...
2026-09-28 20:45:42.851 | INFO     | __main__:main:88 - Number of trainable parameters: 3591
2026-09-28 20:45:42.851 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-28 20:46:01.226 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.679224, Accuracy: 37.50%
2026-09-28 20:46:01.507 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.297883, Accuracy: 62.56%
2026-09-28 20:46:01.797 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.114770, Accuracy: 66.03%
2026-09-28 20:46:02.019 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.851225, Val Accuracy: 70.39%
2026-09-28 20:46:02.020 | INFO 

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

/content/curvature-tuning-research/src/curvature-tuning
2026-09-28 21:17:45.023 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet18_seed43.log
2026-09-28 21:17:45.028 | INFO     | __main__:main:58 - Running on cuda
2026-09-28 21:17:51.765 | INFO     | __main__:main:85 - Testing baseline...
2026-09-28 21:17:51.780 | INFO     | __main__:main:88 - Number of trainable parameters: 3591
2026-09-28 21:17:51.780 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-28 21:18:10.259 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.071714, Accuracy: 3.12%
2026-09-28 21:18:10.547 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.400004, Accuracy: 53.12%
2026-09-28 21:18:10.915 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.158240, Accuracy: 61.74%
2026-09-28 21:18:11.268 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.849565, Val Accuracy: 72.88%
2026-09-28 21:18:11.269 | INFO  

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

/content/curvature-tuning-research/src/curvature-tuning
2026-09-28 21:39:18.197 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet18_seed44.log
2026-09-28 21:39:18.200 | INFO     | __main__:main:58 - Running on cuda
2026-09-28 21:39:24.807 | INFO     | __main__:main:85 - Testing baseline...
2026-09-28 21:39:24.822 | INFO     | __main__:main:88 - Number of trainable parameters: 3591
2026-09-28 21:39:24.822 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-28 21:39:43.372 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.764945, Accuracy: 3.12%
2026-09-28 21:39:43.652 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.716551, Accuracy: 38.95%
2026-09-28 21:39:43.916 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.327978, Accuracy: 54.06%
2026-09-28 21:39:44.119 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.847810, Val Accuracy: 71.29%
2026-09-28 21:39:44.119 | INFO  

### Note on S-CT Numerical Stability

During the DermaMNIST S-CT beta sweeps, some candidate beta values produced non-finite (`NaN`) training losses.

In particular, the highest-curvature-boundary candidates can become numerically unstable.

The original S-CT validation procedure nevertheless completes and selects the best successfully evaluated beta candidate for each seed.

The selected beta values are:

- Seed 42: **0.99**
- Seed 43: **0.82**
- Seed 44: **0.98**

The non-finite candidate runs are retained in the experiment logs as part of the complete experimental record.

# Fit the Frozen Formula-SWCT Predictor

Because this notebook is intended to run independently in a fresh Google Colab runtime, the Formula-SWCT predictor is reconstructed before evaluating DermaMNIST.

The formula is fitted using only the three development/calibration datasets:

| Dataset | Stage 1 | Stage 2 | Stage 3 | Stage 4 |
|---|---:|---:|---:|---:|
| Beans | 0.85 | 0.85 | 0.85 | 0.80 |
| DTD | 0.95 | 0.95 | 0.99 | 0.99 |
| Flowers102 | 0.99 | 0.95 | 0.99 | 0.90 |

For each calibration dataset and seed, five stage descriptors are measured from the frozen ImageNet-pretrained ResNet-18:

1. log activation variance,
2. activation sparsity,
3. log mean absolute activation,
4. log class-separation statistic,
5. normalized stage depth.

A ridge-regression predictor with lambda = 0.10 is fitted from these calibration observations.

DermaMNIST is not used to modify or retune the formula.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode fit \
    --model resnet18 \
    --pretrained_ds imagenet \
    --train_bs 32 \
    --test_bs 64

# Formula-SWCT Evaluation on DermaMNIST

The frozen Formula-SWCT predictor is evaluated on DermaMNIST using seeds 42, 43, and 44.

For each seed:

- stage statistics are collected from the target training split,
- four stage-specific beta values are predicted before classifier training,
- the predicted beta values are frozen,
- features are extracted once,
- one downstream linear classifier is trained.

The fitted formula remains unchanged across all three DermaMNIST runs.

No target-specific beta search or beta optimization is performed.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 20

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: medmnist/dermamnist
Seed: 42
No beta sweep. No beta optimization. No 0.78 anchor.
Generating train split: 7007 examples [00:02, 3388.68 examples/s]
Generating test split: 2005 examples [00:00, 3567.26 examples/s]
Generating validation split: 1003 examples [00:00, 3363.25 examples/s]

Predicted betas BEFORE training:
Stage 1: 0.890654
Stage 2: 0.878392
Stage 3: 0.892478
Stage 4: 0.833190
Predicted beta vector: [0.890654, 0.878392, 0.892478, 0.83319]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.890654, 0.878392, 0.892479, 0.83319]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=0.828675 | val_acc=70.39%
Epoch 02 | val_loss=0.761970 | val_acc=72.28%
Epoch 03 | val_loss=0.734888 | val_acc=72.98%
Epoch 04 | val_loss=0.718903 | val_acc=74.38%
Epoch 05 | val_loss=0.715006 | val_acc=74.28%
Epoch 06 | val_loss=0.6952

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 20

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: medmnist/dermamnist
Seed: 43
No beta sweep. No beta optimization. No 0.78 anchor.

Predicted betas BEFORE training:
Stage 1: 0.890654
Stage 2: 0.878392
Stage 3: 0.892478
Stage 4: 0.833190
Predicted beta vector: [0.890654, 0.878392, 0.892478, 0.83319]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.890654, 0.878392, 0.892479, 0.83319]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=0.854394 | val_acc=70.49%
Epoch 02 | val_loss=0.773900 | val_acc=71.09%
Epoch 03 | val_loss=0.734953 | val_acc=72.28%
Epoch 04 | val_loss=0.712565 | val_acc=73.28%
Epoch 05 | val_loss=0.703633 | val_acc=73.98%
Epoch 06 | val_loss=0.693931 | val_acc=74.68%
Epoch 07 | val_loss=0.688827 | val_acc=74.38%
Epoch 08 | val_loss=0.689175 | val_acc=75.07%
Epoch 09 | val_loss=0.730972 | val_acc=71.98%
Epoch 10 | val_loss=0.672037 | val_acc=74.58

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 20

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: medmnist/dermamnist
Seed: 44
No beta sweep. No beta optimization. No 0.78 anchor.

Predicted betas BEFORE training:
Stage 1: 0.890654
Stage 2: 0.878392
Stage 3: 0.892478
Stage 4: 0.833190
Predicted beta vector: [0.890654, 0.878392, 0.892478, 0.83319]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.890654, 0.878392, 0.892479, 0.83319]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=0.852652 | val_acc=71.39%
Epoch 02 | val_loss=0.766638 | val_acc=72.58%
Epoch 03 | val_loss=0.737327 | val_acc=73.78%
Epoch 04 | val_loss=0.719363 | val_acc=73.88%
Epoch 05 | val_loss=0.703775 | val_acc=74.28%
Epoch 06 | val_loss=0.692298 | val_acc=74.58%
Epoch 07 | val_loss=0.687890 | val_acc=74.58%
Epoch 08 | val_loss=0.684342 | val_acc=74.78%
Epoch 09 | val_loss=0.683297 | val_acc=74.68%
Epoch 10 | val_loss=0.705261 | val_acc=72.98

## Previously Completed Formula-SWCT Runs

All three Formula-SWCT DermaMNIST experiments were previously completed successfully.

The fitted predictor produced the same beta vector across the three seeds:

**[0.890654, 0.878392, 0.892478, 0.833190]**

### Seed 42

- Best validation accuracy: **75.67%**
- Best validation epoch: **16**
- Test accuracy: **75.46%**

### Seed 43

- Best validation accuracy: **75.87%**
- Best validation epoch: **18**
- Test accuracy: **75.21%**

### Seed 44

- Best validation accuracy: **75.97%**
- Best validation epoch: **19**
- Test accuracy: **75.41%**

The Formula-SWCT test results are highly consistent across the three random seeds.

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]

baseline_accs = []
sct_accs = []
sct_betas = []

formula_accs = []
formula_val_accs = []
formula_betas = []

for seed in seeds:

    # Baseline
    with open(
        f"results/base_imagenet_to_medmnist-dermamnist_resnet18_seed{seed}.json",
        "r"
    ) as f:
        baseline = json.load(f)

    # S-CT
    with open(
        f"results/ct_imagenet_to_medmnist-dermamnist_resnet18_seed{seed}.json",
        "r"
    ) as f:
        sct = json.load(f)

    # Formula-SWCT
    with open(
        f"results/formula_swct_imagenet_to_medmnist-dermamnist_resnet18_seed{seed}.json",
        "r"
    ) as f:
        formula = json.load(f)

    baseline_accs.append(baseline["accuracy"])
    sct_accs.append(sct["accuracy"])
    sct_betas.append(sct["beta"])

    formula_accs.append(formula["test_acc"])
    formula_val_accs.append(formula["best_val_acc"])
    formula_betas.append(formula["predicted_betas"])


baseline_accs = np.array(baseline_accs)
sct_accs = np.array(sct_accs)
formula_accs = np.array(formula_accs)
formula_betas = np.array(formula_betas)


print("=" * 70)
print("DERMAMNIST RESULTS — RESNET-18")
print("=" * 70)

print("\nBASELINE")
for seed, acc in zip(seeds, baseline_accs):
    print(f"Seed {seed}: {acc:.2f}%")

print(
    f"Mean: {baseline_accs.mean():.2f}% "
    f"± {baseline_accs.std(ddof=1):.2f}"
)


print("\nS-CT")
for seed, acc, beta in zip(seeds, sct_accs, sct_betas):
    print(
        f"Seed {seed}: "
        f"test={acc:.2f}% | beta={beta:.2f}"
    )

print(
    f"Mean: {sct_accs.mean():.2f}% "
    f"± {sct_accs.std(ddof=1):.2f}"
)


print("\nFORMULA-SWCT")
for seed, acc, val, betas in zip(
    seeds,
    formula_accs,
    formula_val_accs,
    formula_betas
):
    print(
        f"Seed {seed}: "
        f"test={acc:.2f}% | "
        f"best val={val:.2f}% | "
        f"betas={[round(x, 6) for x in betas]}"
    )

print(
    f"\nMean: {formula_accs.mean():.2f}% "
    f"± {formula_accs.std(ddof=1):.2f}"
)

print(
    "\nMean predicted betas:",
    np.round(formula_betas.mean(axis=0), 6)
)

print("\nMETHOD DIFFERENCES")

print(
    f"Formula-SWCT - Baseline: "
    f"{formula_accs.mean() - baseline_accs.mean():+.2f} pp"
)

print(
    f"Formula-SWCT - S-CT: "
    f"{formula_accs.mean() - sct_accs.mean():+.2f} pp"
)

# DermaMNIST Results Summary

Across seeds 42, 43, and 44:

| Method | Test Accuracy |
|---|---:|
| Baseline | **74.83 ± 0.28%** |
| S-CT | **75.28 ± 0.64%** |
| Formula-SWCT | **75.36 ± 0.13%** |

The selected S-CT beta values are:

- Seed 42: **0.99**
- Seed 43: **0.82**
- Seed 44: **0.98**

Formula-SWCT predicts:

**[0.890654, 0.878392, 0.892478, 0.833190]**

across the completed runs.

Relative to the Baseline, Formula-SWCT improves mean test accuracy by approximately:

**+0.53 percentage points**

Relative to S-CT, Formula-SWCT improves mean test accuracy by approximately:

**+0.08 percentage points**

Formula-SWCT therefore achieves comparable mean performance to the target-searched S-CT result while avoiding target-specific beta search and showing lower variation across the three seeds.